In [36]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [37]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
203,This takes place in 1920s Harlem. A black owne...,negative
811,I've always been enthusiastic about period dra...,negative
588,ok we have a film that some are calling one of...,negative
393,I honestly fail to understand why people love ...,negative
35,A ridiculous comedy given an arms-flailing dir...,negative


In [38]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [39]:
df = normalize_text(df)
df.head()


,review,sentiment
203,take place s harlem black owned nightclub deal...,negative
811,always enthusiastic period drama art form bbc ...,negative
588,ok film calling one best movie ever but sittin...,negative
393,honestly fail understand people love show much...,negative
35,ridiculous comedy given arm flailing direction...,negative


In [40]:
df['sentiment'].value_counts()

sentiment
negative    266
positive    234
Name: count, dtype: int64

In [41]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [42]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
203,take place s harlem black owned nightclub deal...,0
811,always enthusiastic period drama art form bbc ...,0
588,ok film calling one best movie ever but sittin...,0
393,honestly fail understand people love show much...,0
35,ridiculous comedy given arm flailing direction...,0


In [43]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [44]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [47]:
import dagshub

mlflow.set_tracking_uri("https://dagshub.com/DataWithPdeep/Capston_Projetc.mlflow")
dagshub.init(repo_owner='DataWithPdeep', repo_name='Capston_Projetc', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=c6814023-c46b-4d93-a807-804e52528974&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=35f33acdd245c49ab6d32043235244b431b7c4296a758b0cb1847cad99464c14




Accessing as DataWithPdeep

Initialized MLflow to track repo "DataWithPdeep/Capston_Projetc"

Repository DataWithPdeep/Capston_Projetc initialized!

2026/06/15 09:30:44 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/fcf210eeab2e49baa175b5ccc5af33b2', creation_time=1781496043831, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781496043831, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [51]:
import mlflow
import logging
import os
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Training Logistic Regression model...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.25)

        logging.info("Fitting the model...")
        model = LogisticRegression()

        logging.info("Fitting the model....")
        model.fit(X_train, y_train)
        logging.info("model fitted successfully.")
        logging.info("Model training complete")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)





2026-06-15 10:49:56,698 - INFO - Starting MLflow run...
2026-06-15 10:50:02,864 - INFO - Training Logistic Regression model...
2026-06-15 10:50:07,938 - INFO - Fitting the model...
2026-06-15 10:50:07,991 - INFO - Fitting the model....
2026-06-15 10:50:08,748 - INFO - model fitted successfully.
2026-06-15 10:50:08,764 - INFO - Model training complete
2026-06-15 10:50:08,764 - INFO - Logging model parameters...
2026-06-15 10:50:09,837 - INFO - Making predictions...
2026-06-15 10:50:09,837 - INFO - Calculating evaluation metrics...
2026-06-15 10:50:09,963 - INFO - Logging evaluation metrics...
2026-06-15 10:50:11,805 - INFO - Saving and logging the model...
2026/06/15 10:50:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 10:50:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbit

🏃 View run chill-kite-749 at: https://dagshub.com/DataWithPdeep/Capston_Projetc.mlflow/#/experiments/0/runs/dde8edc2b83447d499d6bb5d6f599abd
🧪 View experiment at: https://dagshub.com/DataWithPdeep/Capston_Projetc.mlflow/#/experiments/0
